In [6]:
import os
from datasets import load_dataset
from transformers import AutoTokenizer

repo_id = "MrBigBrane/LongBench-v2-32k-CoT"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

# Load a generic background corpus to act as the "haystack" padding
print("Downloading background filler corpus (WikiText)...")
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
filler_text = "\n".join(wiki["text"])
# Tokenizing the entire wikitext corpus gives us a massive pool of ~2M background tokens
global_filler_tokens = tokenizer.encode(filler_text, add_special_tokens=False)

def process_and_pad_example(example, tok, filler_pool, max_tokens=32700):
    import random # <-- Imported locally for Windows multiprocessing workers
    
    question = example.get("question", "")
    choices = (
        f"A) {example.get('choice_A', '')}\n"
        f"B) {example.get('choice_B', '')}\n"
        f"C) {example.get('choice_C', '')}\n"
        f"D) {example.get('choice_D', '')}"
    )
    target = example.get("answer", "")
    
    header = "Read the following text and answer the multiple-choice question. Let's think step-by-step.\n\nBackground Context:\n"
    footer = f"\n\nQuestion: {question}\n{choices}\n\nAnswer:"
    
    header_tokens = tok.encode(header, add_special_tokens=False)
    footer_tokens = tok.encode(footer, add_special_tokens=False)
    
    raw_context = example.get("context", "")
    context_tokens = tok.encode(raw_context, add_special_tokens=False)
    
    # Calculate how much generic padding is needed to hit exactly 32,700
    current_length = len(header_tokens) + len(context_tokens) + len(footer_tokens)
    padding_needed = max_tokens - current_length
    
    if padding_needed > 0:
        # Grab a random contiguous chunk of background tokens to act as padding
        start_idx = random.randint(0, len(filler_pool) - padding_needed - 1)
        padding_tokens = filler_pool[start_idx : start_idx + padding_needed]
        
        # Prepend the padding so the actual context sits closer to the question
        final_tokens = header_tokens + padding_tokens + context_tokens + footer_tokens
    else:
        # If it's naturally over 32,700, cap it safely
        final_tokens = (header_tokens + context_tokens + footer_tokens)[:max_tokens]
        
    final_prompt = tok.decode(final_tokens)
    
    return {
        "input": final_prompt,
        "outputs": [target],
        "padded_tokens_added": padding_needed if padding_needed > 0 else 0
    }

def filter_short(example, tok):
    # Only keep examples under 30,000 tokens so we can pad them UP
    return len(tok.encode(example["context"], add_special_tokens=False)) < 30000

if __name__ == "__main__":
    num_cpus = max(1, os.cpu_count() - 2)
    print(f"Downloading LongBench-v2...")
    dataset = load_dataset("THUDM/LongBench-v2", split="train")

    print("Filtering dataset for examples under 30k tokens...")
    dataset = dataset.filter(
        filter_short,
        fn_kwargs={"tok": tokenizer},
        num_proc=num_cpus
    )

    print(f"Padding {len(dataset)} examples to exactly 32,700 tokens...")
    formatted = dataset.map(
        process_and_pad_example,
        fn_kwargs={
            "tok": tokenizer, 
            "filler_pool": global_filler_tokens,
            "max_tokens": 32700
        },
        remove_columns=dataset.column_names,
        num_proc=num_cpus
    )

    formatted.push_to_hub(repo_id, split="train")
    print(f"Successfully uploaded {len(formatted)} padded examples to {repo_id}!")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2517233 > 131072). Running this sequence through the model will result in indexing errors


Filtering dataset for examples under 30k tokens...
Padding 106 examples to exactly 32,700 tokens...


Map (num_proc=22): 100%|██████████| 106/106 [01:35<00:00,  1.10 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 11.82ba/s]
Processing Files (1 / 1): 100%|██████████| 7.33MB / 7.33MB,  559kB/s  
New Data Upload: 100%|██████████| 7.33MB / 7.33MB,  559kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:04<00:00,  4.25s/ shards]


Successfully uploaded 106 padded examples to MrBigBrane/LongBench-v2-32k-CoT!


In [8]:
from datasets import load_dataset

# Force a fresh download to overwrite the cached schema
test_dataset = load_dataset(
    "MrBigBrane/LongBench-v2-32k-CoT", 
    split="train",
    download_mode="force_redownload"
)

print(f"Total examples ready for sweep: {len(test_dataset)}")
print(f"Sample prompt length (characters): {len(test_dataset[0]['input'])}")
print(f"Padded tokens added to first example: {test_dataset[0]['padded_tokens_added']}")

Generating train split:   0%|          | 0/106 [00:00<?, ? examples/s]Failed to read file 'C:\Users\theep\.cache\huggingface\hub\datasets--MrBigBrane--LongBench-v2-32k-CoT\snapshots\ace22e6650eb7c499e29d2ced45a93adec6a7061\data\train-00000-of-00001.parquet' with error CastError: Couldn't cast
input: string
outputs: list<element: string>
  child 0, element: string
padded_tokens_added: int64
-- schema metadata --
huggingface: '{"info": {"features": {"input": {"dtype": "string", "_type"' + 154
to
{'input': Value('string'), 'outputs': List(Value('string')), 'original_token_length': Value('int64')}
because column names don't match
Generating train split:   0%|          | 0/106 [00:00<?, ? examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [10]:
from datasets import load_dataset

# Load the raw Parquet file directly, ignoring any cached repository schema
test_dataset = load_dataset(
    "parquet", 
    data_files="hf://datasets/MrBigBrane/LongBench-v2-32k-CoT/data/train-*", 
    split="train"
)

print(f"Total examples ready for sweep: {len(test_dataset)}")
print(f"Sample prompt length (characters): {len(test_dataset[0]['input'])}")
print(f"Padded tokens added to first example: {test_dataset[0]['padded_tokens_added']}")

Generating train split: 106 examples [00:00, 4704.17 examples/s]

Total examples ready for sweep: 106
Sample prompt length (characters): 135072
Padded tokens added to first example: 8535


In [11]:
from datasets import load_dataset

repo_id = "MrBigBrane/LongBench-v2-32k-CoT"

# 1. Load the parquet file directly to bypass the schema conflict
ds = load_dataset(
    "parquet",
    data_files="hf://datasets/MrBigBrane/LongBench-v2-32k-CoT/data/train-*",
    split="train",
)

# 2. Keep only the exact columns expected by RULER ('input' and 'outputs')
ds = ds.remove_columns([c for c in ds.column_names if c not in ["input", "outputs"]])

# 3. Push to hub under the subset/config name matching your task: 'longbench_v2_cot'
ds.push_to_hub(repo_id, config_name="longbench_v2_cot", split="train")

print("Uploaded successfully under subset 'longbench_v2_cot'!")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 16.11ba/s]
Processing Files (1 / 1): 100%|██████████| 7.33MB / 7.33MB,  669kB/s  
New Data Upload: 100%|██████████| 21.3kB / 21.3kB, 2.00kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.43s/ shards]


Uploaded successfully under subset 'longbench_v2_cot'!


In [12]:
import os
from datasets import load_dataset

REPO_ID = "MrBigBrane/LongBench-v2-32k-CoT"
CONFIG_NAME = "longbench_v2_cot"

# Conclude concisely and enforce the exact final extraction string
INSTRUCTION_SUFFIX = (
    "\n\nKeep your reasoning concise (under 3 sentences). "
    "On the final line, provide your answer in the exact format: 'Final Answer: [A/B/C/D]'."
)

print(f"Loading dataset '{REPO_ID}' (config: {CONFIG_NAME})...")
ds = load_dataset(REPO_ID, CONFIG_NAME)


def append_instruction(example):
  raw_input = example["input"].rstrip()
  example["input"] = raw_input + INSTRUCTION_SUFFIX
  return example


print("Appending concise conclusion instructions...")
updated_ds = ds.map(append_instruction)

# Verification check on the first row
sample_prompt = updated_ds["train"][0]["input"]
print("\n=== VERIFICATION: Last 300 characters of prompt #0 ===")
print(sample_prompt[-300:])
print("=" * 60)

# Push update back to Hugging Face
print(f"\nPushing updated dataset back to '{REPO_ID}'...")
updated_ds.push_to_hub(
    repo_id=REPO_ID,
    config_name=CONFIG_NAME,
    commit_message="Enforce concise reasoning limit and Final Answer string format",
)

print("Dataset successfully updated on Hugging Face Hub!")

Loading dataset 'MrBigBrane/LongBench-v2-32k-CoT' (config: longbench_v2_cot)...
Appending concise conclusion instructions...


Map: 100%|██████████| 106/106 [00:00<00:00, 698.36 examples/s]


=== VERIFICATION: Last 300 characters of prompt #0 ===
 It is the readers' obligation to get the "truth" from the primary narrator.
D) The performative interpretation of language transforms what it interprets.

Answer:

Keep your reasoning concise (under 3 sentences). On the final line, provide your answer in the exact format: 'Final Answer: [A/B/C/D]'.

Pushing updated dataset back to 'MrBigBrane/LongBench-v2-32k-CoT'...



Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 23.67ba/s]
Processing Files (1 / 1): 100%|██████████| 7.34MB / 7.34MB,  604kB/s  
New Data Upload: 100%|██████████| 7.15MB / 7.15MB,  589kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.37s/ shards]


Dataset successfully updated on Hugging Face Hub!


In [1]:
import os
import re
from datasets import load_dataset

REPO_ID = "MrBigBrane/LongBench-v2-32k-CoT"
CONFIG_NAME = "longbench_v2_cot"

# Regex matching the previous suffix so we replace it cleanly without accumulating duplicate instructions
OLD_INSTRUCTION_PATTERN = re.compile(
    r"\n\nKeep your reasoning concise \(under 3 sentences\)\..*?Final Answer:.*$",
    re.DOTALL,
)

# New strict single-choice constraint
NEW_INSTRUCTION_SUFFIX = (
    "\n\nKeep your reasoning concise (under 3 sentences). "
    "There is exactly one correct option. "
    "On the final line, provide your answer in the exact format: "
    "'Final Answer: [X]' where X is a single letter (A, B, C, or D)."
)

print(f"Loading dataset '{REPO_ID}' (config: {CONFIG_NAME})...")
ds = load_dataset(REPO_ID, CONFIG_NAME)


def update_instruction(example):
  raw_input = example["input"].rstrip()
  # Strip existing prompt instruction if present, then append the new one
  cleaned_input = OLD_INSTRUCTION_PATTERN.sub("", raw_input).rstrip()
  example["input"] = cleaned_input + NEW_INSTRUCTION_SUFFIX
  return example


print("Updating prompt instructions with single-choice constraint...")
updated_ds = ds.map(update_instruction)

# Verification check on prompt #0
sample_prompt = updated_ds["train"][0]["input"]
print("\n=== VERIFICATION: Last 350 characters of prompt #0 ===")
print(sample_prompt[-350:])
print("=" * 60)

# Push update back to Hugging Face
print(f"\nPushing updated dataset back to '{REPO_ID}'...")
updated_ds.push_to_hub(
    repo_id=REPO_ID,
    config_name=CONFIG_NAME,
    commit_message="Enforce single-choice constraint on Final Answer format",
)

print("Dataset successfully updated on Hugging Face Hub!")

c:\CondaEnvs\opus-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset 'MrBigBrane/LongBench-v2-32k-CoT' (config: longbench_v2_cot)...


c:\CondaEnvs\opus-ai\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\theep\.cache\huggingface\hub\datasets--MrBigBrane--LongBench-v2-32k-CoT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 106/106 [00:00<00:00, 1821.85 examples/s]


Updating prompt instructions with single-choice constraint...


Map: 100%|██████████| 106/106 [00:00<00:00, 1760.29 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.



=== VERIFICATION: Last 350 characters of prompt #0 ===
gation to get the "truth" from the primary narrator.
D) The performative interpretation of language transforms what it interprets.

Answer:

Keep your reasoning concise (under 3 sentences). There is exactly one correct option. On the final line, provide your answer in the exact format: 'Final Answer: [X]' where X is a single letter (A, B, C, or D).

Pushing updated dataset back to 'MrBigBrane/LongBench-v2-32k-CoT'...


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 25.89ba/s]
Processing Files (1 / 1): 100%|██████████| 7.36MB / 7.36MB,  600kB/s  
New Data Upload: 100%|██████████| 7.17MB / 7.17MB,  585kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.56s/ shards]


Dataset successfully updated on Hugging Face Hub!


In [2]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

REPO_ID = "MrBigBrane/LongBench-v2-32k-CoT"
CONFIG_NAME = "longbench_v2_cot"
TOKENIZER_MODEL = "Qwen/Qwen2.5-7B-Instruct"
TOKENIZER_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"

print(f"Loading tokenizer '{TOKENIZER_MODEL}'...")
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_MODEL, revision=TOKENIZER_REVISION, trust_remote_code=True
)

print(f"Loading dataset '{REPO_ID}'...")
ds = load_dataset(REPO_ID, CONFIG_NAME, split="train")

token_lengths = []
per_prompt_details = []

print(f"Profiling token lengths across {len(ds)} prompts...\n")

for i, example in enumerate(ds):
  prompt_text = example.get("input", "")

  # Verify using apply_chat_template if your harness wraps inputs as chat
  # If the harness feeds raw text, direct tokenization is used
  tokens = tokenizer.encode(prompt_text, add_special_tokens=False)
  num_tokens = len(tokens)
  token_lengths.append(num_tokens)

  idx = example.get("index", example.get("id", i))
  per_prompt_details.append((idx, num_tokens))

# Sort to identify outliers
per_prompt_details.sort(key=lambda x: x[1])

print(f"{'Prompt Index':<15}{'Token Count':<15}")
print("-" * 30)
for idx, count in per_prompt_details[:5]:
  print(f"{idx:<15}{count:<15}")
if len(per_prompt_details) > 10:
  print(f"{'...':<15}{'...':<15}")
for idx, count in per_prompt_details[-5:]:
  print(f"{idx:<15}{count:<15}")

lengths_arr = np.array(token_lengths)
print("\n" + "=" * 35)
print("       TOKEN LENGTH AUDIT")
print("=" * 35)
print(f"Total Prompts:       {len(token_lengths)}")
print(f"Min Tokens:          {lengths_arr.min()}")
print(f"Max Tokens:          {lengths_arr.max()}")
print(f"Mean Tokens:         {lengths_arr.mean():.1f}")
print(f"Median Tokens:       {np.median(lengths_arr):.1f}")
print(f"Std Deviation:       {lengths_arr.std():.2f}")
print(f"Prompts > 32768:     {(lengths_arr > 32768).sum()}")
print(
    f"Prompts in [32k, 33k]: {((lengths_arr >= 32000) & (lengths_arr <= 33000)).sum()}"
)
print("=" * 35)

Loading tokenizer 'Qwen/Qwen2.5-7B-Instruct'...


c:\CondaEnvs\opus-ai\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\theep\.cache\huggingface\hub\models--Qwen--Qwen2.5-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading dataset 'MrBigBrane/LongBench-v2-32k-CoT'...


Generating train split: 100%|██████████| 106/106 [00:00<00:00, 3693.06 examples/s]


Profiling token lengths across 106 prompts...

Prompt Index   Token Count    
------------------------------
58             32751          
60             32751          
0              32752          
1              32752          
3              32752          
...            ...            
96             32753          
98             32753          
101            32753          
104            32753          
105            32753          

       TOKEN LENGTH AUDIT
Total Prompts:       106
Min Tokens:          32751
Max Tokens:          32753
Mean Tokens:         32752.4
Median Tokens:       32752.0
Std Deviation:       0.52
Prompts > 32768:     0
Prompts in [32k, 33k]: 106


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load tokenizer for your target base architecture
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# Load LongBench-v2 short
raw_dataset = load_dataset("THUDM/LongBench-v2", split="train")
short_subset = raw_dataset.filter(lambda example: example["length"] == "short")

NEW_INSTRUCTION_SUFFIX = (
    "\n\nKeep your reasoning concise (under 3 sentences). "
    "There is exactly one correct option. "
    "On the final line, provide your answer in the exact format: "
    "'Final Answer: [X]' where X is a single letter (A, B, C, or D)."
)

def add_templated_prompt(example):
    user_content = (
        f"{example['context']}\n\n"
        f"Question: {example['question']}\n"
        f"A. {example['choice_A']}\n"
        f"B. {example['choice_B']}\n"
        f"C. {example['choice_C']}\n"
        f"D. {example['choice_D']}"
        f"{NEW_INSTRUCTION_SUFFIX}"
    )
    
    messages = [{"role": "user", "content": user_content}]
    
    # tokenize=False returns raw string with chat tokens appended
    # add_generation_prompt=True appends the assistant start tokens (e.g., <|start_header_id|>assistant<|end_header_id|>)
    templated_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    return {
        "raw_prompt": user_content,
        "templated_prompt": templated_str
    }

formatted_ds = short_subset.map(add_templated_prompt)

# Push to Hub
formatted_ds.push_to_hub("MrBigBrane/longbench-v2-short")

Map: 100%|██████████| 180/180 [00:00<00:00, 820.50 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  6.72ba/s]
Processing Files (1 / 1): 100%|██████████| 29.1MB / 29.1MB, 1.73MB/s  
New Data Upload: 100%|██████████| 28.8MB / 28.8MB, 1.72MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:08<00:00,  8.99s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/MrBigBrane/longbench-v2-short/commit/3f253259aa6b9879f2e7ad30823e7876e18fe634', commit_message='Upload dataset', commit_description='', oid='3f253259aa6b9879f2e7ad30823e7876e18fe634', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MrBigBrane/longbench-v2-short', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MrBigBrane/longbench-v2-short'), pr_revision=None, pr_num=None)

In [4]:
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer

# 1. Model tokenizer used for formatting chat tokens
# Change to your target model architecture (e.g., Qwen/Qwen2.5-7B-Instruct, etc.)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 2. Load LongBench-v2 and isolate the short tier
raw_ds = load_dataset("THUDM/LongBench-v2", split="train")
short_ds = raw_ds.filter(lambda x: x["length"] == "short")

NEW_INSTRUCTION_SUFFIX = (
    "\n\nKeep your reasoning concise (under 3 sentences). "
    "There is exactly one correct option. "
    "On the final line, provide your answer in the exact format: "
    "'Final Answer: [X]' where X is a single letter (A, B, C, or D)."
)


def process_row(example):
    # Assemble the raw prompt text
    user_content = (
        f"{example['context']}\n\n"
        f"Question: {example['question']}\n"
        f"A. {example['choice_A']}\n"
        f"B. {example['choice_B']}\n"
        f"C. {example['choice_C']}\n"
        f"D. {example['choice_D']}"
        f"{NEW_INSTRUCTION_SUFFIX}"
    )

    # Wrap in standard chat structure and append generation prompt tokens
    chat_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "inputs": chat_prompt,
        "outputs": example["answer"].strip(),  # 'A', 'B', 'C', or 'D'
        "difficulty": example.get("difficulty", "unknown"),
        "domain": example.get("domain", "unknown"),
        "sub_domain": example.get("sub_domain", "unknown"),
    }


# Map and drop all raw intermediate columns
processed_ds = short_ds.map(
    process_row, remove_columns=raw_ds.column_names, desc="Formatting dataset"
)

# 3. Push to Hugging Face
REPO_ID = "MrBigBrane/longbench-v2-short"
processed_ds.push_to_hub(
    repo_id=REPO_ID,
    private=False,
    commit_message="Add formatted inputs/outputs with metadata columns",
)

print(f"Uploaded successfully to: https://huggingface.co/datasets/{REPO_ID}")

Formatting dataset: 100%|██████████| 180/180 [00:00<00:00, 1916.89 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 18.30ba/s]
Processing Files (1 / 1): 100%|██████████| 9.71MB / 9.71MB,  893kB/s  
New Data Upload: 100%|██████████|  141kB /  141kB, 13.4kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.52s/ shards]


Uploaded successfully to: https://huggingface.co/datasets/MrBigBrane/longbench-v2-short
